# Phase 3: Logistic Regression Scorecard Model
In this notebook, we build the champion credit scorecard. We train a Logistic Regression model on the selected WoE features, check model coefficients and multicollinearity (VIF), and scale log-odds into score points.


In [1]:
import pandas as pd
import os
import sys
import yaml

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from woe_binning import WoEBinning
from scorecard import ScorecardModel


## 1. Load Data & Mappings
Load the dataset splits and the WoE mapping object.


In [2]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()

woe_model = WoEBinning(target_col='target')
woe_model.fit_all(train_df, [
    'loan_amnt', 'annual_inc', 'dti', 'revol_util', 
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 
    'pub_rec', 'pub_rec_bankruptcies', 'credit_history_age',
    'emp_length'
], ['home_ownership', 'purpose'])

train_woe = woe_model.transform(train_df)
selected_woe_cols = [f'{col}_woe' for col in woe_model.selected_features]


Fitting Numeric Columns...
  Fitting loan_amnt...
  Fitting annual_inc...
  Fitting dti...
  Fitting revol_util...
  Fitting delinq_2yrs...
  Fitting inq_last_6mths...
  Fitting open_acc...
  Fitting pub_rec...
  Fitting pub_rec_bankruptcies...
  Fitting credit_history_age...
  Fitting emp_length...
Fitting Categorical Columns...
  Fitting home_ownership...
  Fitting purpose...


## 2. Train Logistic Regression
We train the regression model predicting default (target=1) using statsmodels.


In [3]:
scorecard = ScorecardModel(base_score=600, base_odds=50, pdo=20)
scorecard.fit(train_woe[selected_woe_cols], train_woe['target'])
print(scorecard.model.summary())


                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:                18061
Model:                          Logit   Df Residuals:                    18054
Method:                           MLE   Df Model:                            6
Date:                Mon, 15 Jun 2026   Pseudo R-squ.:                 0.04084
Time:                        19:04:30   Log-Likelihood:                -6735.5
converged:                       True   LL-Null:                       -7022.3
Covariance Type:            nonrobust   LLR p-value:                1.153e-120
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       -1.8902      0.023    -82.963      0.000      -1.935      -1.846
revol_util_woe              -1.0537      0.081    -13.046      0.000      -1.212      -0.

## 3. Multicollinearity & Diagnostic Checks
We check Variance Inflation Factors (VIF) and verify that all coefficients are statistically significant (p < 0.05) and have the expected sign (negative for default prediction).


In [4]:
print('Variance Inflation Factor (VIF) Results:')
print(scorecard.vif_report)

is_valid, issues = scorecard.validate_coefficients()
print('\nModel Diagnostics Check:')
if is_valid:
    print('SUCCESS: All coefficient signs are correct and VIF values are within regulatory limits.')
else:
    print('WARNING Issues Detected:')
    for issue in issues:
        print(' -', issue)


Variance Inflation Factor (VIF) Results:
                feature       VIF
4               pub_rec  2.376652
5  pub_rec_bankruptcies  2.372423
0            revol_util  1.008176
2        inq_last_6mths  1.005976
3            annual_inc  1.004245
1               purpose  1.004242

Model Diagnostics Check:
WARNING Issues Detected:
 - Feature 'pub_rec_bankruptcies_woe' is not statistically significant (p-value = 0.4616 >= 0.05).


## 4. Scorecard Scaling & Points Mapping
Translate the logistic regression coefficients into integer-based score contributions per bin.


In [5]:
scorecard.build_scorecard_table(woe_model.mappings)
scorecard_table = scorecard.scorecard_table
print('Scorecard Points Table (First 15 Rows):')
print(scorecard_table.head(15))


Scorecard Points Table (First 15 Rows):
          Variable                                                Bin  \
0       revol_util                                            <= 1.89   
1       revol_util                                      (1.89, 27.55]   
2       revol_util                                     (27.55, 38.05]   
3       revol_util                                     (38.05, 80.25]   
4       revol_util                                            > 80.25   
5       revol_util                                            Missing   
6          purpose  wedding, credit_card, car, major_purchase, hom...   
7          purpose       debt_consolidation, medical, other, vacation   
8          purpose  educational, house, renewable_energy, small_bu...   
9   inq_last_6mths                                               <= 0   
10  inq_last_6mths                                             (0, 1]   
11  inq_last_6mths                                             (1, 2]   
12  inq_las

## 5. Customer Score Generation
Demonstrate credit scoring on train sample and show the output scores and PD values.


In [6]:
scores = scorecard.predict_score(train_woe)
pds = scorecard.predict_pd(train_woe)
scored_df = pd.DataFrame({'Target': train_woe['target'], 'Credit Score': scores, 'PD': pds})
print(scored_df.describe())


             Target  Credit Score            PD
count  18061.000000  18061.000000  18061.000000
mean       0.131277    544.513759      0.131277
std        0.337713     15.117256      0.062790
min        0.000000    474.000000      0.036679
25%        0.000000    535.000000      0.088504
50%        0.000000    545.000000      0.118746
75%        0.000000    554.000000      0.160437
max        1.000000    581.000000      0.609660
